# Graph configurations

Inventory of every network realization used by the experiments.

**The unit of analysis is the realization (`graph_id`), not the config.** The two base
configs (`erdos-renyi`, `watts-strogatz`) each contain eight distinct graphs drawn from the same
generative parameters; every maximin-selected config contains one. Collapsing them by config name hides
eight-fold structural variation, so each graph is kept separate throughout and given a `label`
of the form `erdos-renyi#3`.

Four sources are joined:

| source | what it gives | key |
|---|---|---|
| `src/configs/network/*.yaml` | generator and parameters (`p` for ER, `k`/`beta` for WS) | `seed_map = {master_seed: graph_seed}` |
| `data/resources/seeds/graph_manifest_{er,ws}.csv` | precomputed design stats, `selection_order`, `d_min` | `seed` = graph seed |
| `data/outputs/runs/**/results/network_edges.json` | the adjacency matrix itself | `graph_type` + `seed` |
| `data/outputs/aggregated_expanded/agent_level_data.parquet` | run coverage, node-level metrics | `graph_id` |

**Join caveat.** The parquet's `graph_seed` and the run metadata's `seed` are the **master**
seed, not the manifest's `seed` column (`erdos-renyi-01-1866808230` ↔ YAML key `1866808230`,
whose value `484618803` is the graph seed). The cells below assert the row count after each merge.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import yaml
import networkx as nx
import matplotlib.pyplot as plt
from networkx.algorithms.community import greedy_modularity_communities, modularity

REPO_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(Path.cwd()))

from exp_plot_utils import _configure_fonts, GRAPH_COLOR, OKABE_ITO
from src.analysis.loaders import load_runs_metadata, load_adjacency_matrix

_configure_fonts()

CONFIG_DIR = REPO_ROOT / "src" / "configs" / "network"
MANIFEST_DIR = REPO_ROOT / "data" / "resources" / "seeds"
RUNS_DIR = REPO_ROOT / "data" / "outputs" / "runs"
PARQUET = (
    REPO_ROOT / "data" / "outputs" / "aggregated_expanded" / "agent_level_data.parquet"
)

out_path = REPO_ROOT / "data" / "analysis" / "expanded" / "graphs"
out_path.mkdir(parents=True, exist_ok=True)

## 1. Configs to realizations

One row per realization. `is_origin` marks the two multi-seed base configs.

In [ ]:
rows = []
for f in sorted(CONFIG_DIR.glob("*.yaml")):
    if f.stem == "base":  # abstract parent, no generator
        continue
    d = yaml.safe_load(f.read_text())
    for master_seed, graph_seed in (d.get("seed_map") or {}).items():
        rows.append(
            {
                "config": f.stem,
                "generator": d["generator"],
                "master_seed": int(master_seed),
                "graph_seed": int(graph_seed),
                **d["params"],
            }
        )

cfg = pd.DataFrame(rows)
cfg["family"] = cfg["config"].str.replace(r"-\d{2}$", "", regex=True)
cfg["is_origin"] = ~cfg["config"].str.contains(r"-\d{2}$", regex=True)
cfg["graph_id"] = cfg["config"] + "-" + cfg["master_seed"].astype(str)

# Distinguish the eight realizations inside each original config.
cfg = cfg.sort_values(["config", "master_seed"]).reset_index(drop=True)
cfg["realization"] = cfg.groupby("config").cumcount() + 1
n_per_config = cfg.groupby("config")["graph_id"].transform("size")
cfg["label"] = np.where(
    n_per_config > 1,
    cfg["config"] + "#" + cfg["realization"].astype(str),
    cfg["config"],
)
cfg["short"] = (
    cfg["label"]
    .str.replace("watts-strogatz", "WS", regex=False)
    .str.replace("erdos-renyi", "ER", regex=False)
)

print(f"configs: {cfg['config'].nunique()}   realizations: {len(cfg)}")
print(cfg.groupby("config").size().to_string())

## 2. Join the design manifests

In [ ]:
manifest = pd.concat(
    [
        pd.read_csv(MANIFEST_DIR / "graph_manifest_er.csv"),
        pd.read_csv(MANIFEST_DIR / "graph_manifest_ws.csv"),
    ],
    ignore_index=True,
)

graphs = cfg.merge(
    manifest[["seed", "master_seed", "selection_order", "d_min"]],
    left_on=["graph_seed", "master_seed"],
    right_on=["seed", "master_seed"],
    how="left",
).drop(columns="seed")

assert len(graphs) == len(cfg), "merge duplicated rows"
print(
    f"{len(graphs)} realizations, {graphs['selection_order'].notna().sum()} carry a maximin selection_order"
)

## 3. Structural features from the adjacency matrices

The manifest only stores mean degree, clustering, mean shortest path and diameter. Everything
else is computed here from `network_edges.json`. One run directory per realization is enough,
since the topology is fixed across scenarios and statements.

Centralization is Freeman's index: how far the graph is from a star, given the observed spread
of a centrality. Degree centralization uses the exact star normalization `(n-1)(n-2)`; for the
already-normalized centralities the generic `sum(max - v) / (n - 1)` is reported.

In [ ]:
def _gini(x):
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    if n == 0 or x.sum() == 0:
        return 0.0
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum()))


def _centralization(values, n):
    """Generic Freeman centralization for an already-normalized centrality."""
    v = np.asarray(values, dtype=float)
    return float((v.max() - v).sum() / (n - 1)) if n > 1 else 0.0


def graph_features(A: np.ndarray) -> dict:
    """Structural summary of one undirected, unweighted adjacency matrix."""
    G = nx.from_numpy_array(A)
    n = G.number_of_nodes()
    deg = np.array([d for _, d in G.degree()], dtype=float)

    connected = nx.is_connected(G)
    comps = list(nx.connected_components(G))
    Gc = G.subgraph(max(comps, key=len)).copy()  # largest component

    btw = nx.betweenness_centrality(G)
    clo = nx.closeness_centrality(G)
    eig = nx.eigenvector_centrality_numpy(G)
    pr = nx.pagerank(G)
    core = nx.core_number(G)
    comms = greedy_modularity_communities(G)

    # Adjacency spectrum: the gap lambda_1 - lambda_2 bounds mixing speed.
    lam = np.sort(np.linalg.eigvalsh(A.astype(float)))[::-1]

    return {
        # size / density
        "n": n,
        "m": G.number_of_edges(),
        "density": nx.density(G),
        "mean_deg": deg.mean(),
        "std_deg": deg.std(ddof=0),
        "min_deg": deg.min(),
        "max_deg": deg.max(),
        "deg_gini": _gini(deg),
        # Undefined when every node has the same degree (an unrewired ring lattice):
        # the coefficient divides by the degree variance. Report 0.0, not NaN.
        "deg_assortativity": (
            0.0 if deg.std() == 0 else nx.degree_assortativity_coefficient(G)
        ),
        # clustering
        "gcc": nx.transitivity(G),  # global clustering coefficient
        "avg_clustering": nx.average_clustering(G),
        # distance (on the largest component, so disconnected graphs stay comparable)
        "connected": connected,
        "n_components": len(comps),
        "largest_cc_frac": Gc.number_of_nodes() / n,
        "mean_sp": nx.average_shortest_path_length(Gc),
        "diameter": nx.diameter(Gc),
        "radius": nx.radius(Gc),
        # centrality spread
        "btw_mean": np.mean(list(btw.values())),
        "btw_max": np.max(list(btw.values())),
        "eig_sd": np.std(list(eig.values()), ddof=0),
        "eig_max": np.max(list(eig.values())),
        "clo_mean": np.mean(list(clo.values())),
        "pagerank_gini": _gini(list(pr.values())),
        "centralization_deg": float((deg.max() - deg).sum() / ((n - 1) * (n - 2))),
        "centralization_btw": _centralization(list(btw.values()), n),
        "centralization_eig": _centralization(list(eig.values()), n),
        # mesoscale / spectral
        "modularity": modularity(G, comms),
        "n_communities": len(comms),
        "max_core": max(core.values()),
        "algebraic_connectivity": nx.algebraic_connectivity(Gc),
        "spectral_gap": float(lam[0] - lam[1]),
    }

In [ ]:
runs = load_runs_metadata(RUNS_DIR)
runs = runs[runs["validated"] & runs["completed"]].copy()
runs["graph_id"] = runs["graph_type"].astype(str) + "-" + runs["seed"].astype(str)

# One representative run per realization: topology is fixed across scenarios and statements.
representative = runs.groupby("graph_id", as_index=False).first()[
    ["graph_id", "run_path"]
]

feat_rows = []
for gid, run_path in representative.itertuples(index=False):
    A, _ = load_adjacency_matrix(run_path)
    feat_rows.append({"graph_id": gid, **graph_features(A)})

features = pd.DataFrame(feat_rows)
print(f"computed {features.shape[1] - 1} features for {len(features)} realizations")

### Cross-check against the manifest and the parquet

In [ ]:
graphs = graphs.merge(features, on="graph_id", how="left")
assert graphs["mean_deg"].notna().all(), "realization missing an adjacency matrix"

par = pl.read_parquet(PARQUET)
coverage = (
    par.group_by("graph_id")
    .agg(pl.col("run_id").n_unique().alias("n_runs"))
    .to_pandas()
)
graphs = graphs.merge(coverage, on="graph_id", how="left")
graphs["n_runs"] = graphs["n_runs"].fillna(0).astype(int)

print(
    "configured but never run :",
    graphs.loc[graphs["n_runs"] == 0, "graph_id"].tolist() or "none",
)
print(
    "run but not configured   :",
    sorted(set(coverage["graph_id"]) - set(graphs["graph_id"])) or "none",
)

# Degrees recomputed from the adjacency must match those recorded per agent in the runs.
realised = (
    par.unique(subset=["graph_id", "agent_id"])
    .group_by("graph_id")
    .agg(pl.col("node_degree").mean().alias("deg_parquet"))
    .to_pandas()
)
chk = graphs.merge(realised, on="graph_id")
print(
    f"max |adjacency mean_deg - parquet mean_deg| = {(chk['mean_deg'] - chk['deg_parquet']).abs().max():.2e}"
)

man_chk = cfg.merge(
    manifest[["seed", "master_seed", "mean_deg", "gcc"]],
    left_on=["graph_seed", "master_seed"],
    right_on=["seed", "master_seed"],
)
man_chk = man_chk.merge(
    features[["graph_id", "mean_deg", "gcc"]], on="graph_id", suffixes=("_man", "_adj")
)
print(
    f"max |manifest mean_deg - adjacency mean_deg| = {(man_chk['mean_deg_man'] - man_chk['mean_deg_adj']).abs().max():.2e}"
)
print(
    f"max |manifest gcc      - adjacency gcc     | = {(man_chk['gcc_man'] - man_chk['gcc_adj']).abs().max():.2e}"
)

## 4. The realization table

`selection_order` and `d_min` come from the maximin design in
`notebooks/maximin_selection.ipynb`: each maximin-selected realization was chosen to be
maximally distant, in structural-feature space, from everything already selected. `d_min` is that
distance at the moment of selection, so it decreases down the order. The 16 base realizations
are the procedure's starting set, so these columns are blank for them.

In [ ]:
COLS = [
    "label",
    "family",
    "config",
    "realization",
    "master_seed",
    "graph_seed",
    "p",
    "k",
    "beta",
    "n",
    "m",
    "density",
    "mean_deg",
    "std_deg",
    "min_deg",
    "max_deg",
    "deg_gini",
    "deg_assortativity",
    "gcc",
    "avg_clustering",
    "connected",
    "n_components",
    "mean_sp",
    "diameter",
    "radius",
    "modularity",
    "n_communities",
    "max_core",
    "algebraic_connectivity",
    "spectral_gap",
    "centralization_deg",
    "centralization_btw",
    "centralization_eig",
    "btw_max",
    "eig_sd",
    "eig_max",
    "clo_mean",
    "pagerank_gini",
    "selection_order",
    "d_min",
    "n_runs",
]
COLS = [c for c in COLS if c in graphs.columns]

table = graphs[COLS].sort_values(["family", "mean_deg"]).reset_index(drop=True)
table.to_csv(out_path / "graph_configurations.csv", index=False)

pd.set_option("display.max_columns", None, "display.width", 250)
table.round(3)

### Within the base configs

The eight realizations of each base config are *not* interchangeable. Grouping by config name
hides this spread.

In [ ]:
origin_cols = [
    "label",
    "mean_deg",
    "std_deg",
    "gcc",
    "mean_sp",
    "diameter",
    "modularity",
    "deg_assortativity",
    "centralization_deg",
    "algebraic_connectivity",
]
orig = table[table["config"].isin(["erdos-renyi", "watts-strogatz"])]
display(orig[origin_cols].round(3))

spread = (
    orig.assign(config=orig["label"].str.split("#").str[0])
    .groupby("config")[
        ["mean_deg", "gcc", "mean_sp", "modularity", "centralization_deg"]
    ]
    .agg(["min", "max", "std"])
)
print("\nrange across the eight realizations of each original config:")
spread.round(3)

## 5. Design coverage

Each point is one realization. Filled circles are the base configs, open triangles the
maximin-selected configs.

In [ ]:
PANELS = [
    ("mean_deg", "gcc", "Mean degree", "Global clustering"),
    ("mean_deg", "mean_sp", "Mean degree", "Mean shortest path"),
    ("mean_deg", "modularity", "Mean degree", "Modularity"),
    ("gcc", "deg_assortativity", "Global clustering", "Degree assortativity"),
    ("mean_deg", "centralization_deg", "Mean degree", "Degree centralization"),
    ("mean_deg", "algebraic_connectivity", "Mean degree", "Algebraic connectivity"),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 7.5))
for ax, (x, y, xl, yl) in zip(axes.ravel(), PANELS):
    for fam, sub in graphs.groupby("family"):
        for is_origin, s in sub.groupby("is_origin"):
            ax.scatter(
                s[x],
                s[y],
                s=75 if is_origin else 55,
                marker="o" if is_origin else "^",
                facecolor=GRAPH_COLOR[fam] if is_origin else "none",
                edgecolor=GRAPH_COLOR[fam],
                linewidth=1.4,
                alpha=0.9,
                label=f"{fam} ({'original' if is_origin else 'extension'})",
            )
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)

axes[0, 0].legend(frameon=False, fontsize=8, loc="upper left")
fig.tight_layout()
fig.savefig(out_path / "design_coverage.pdf", dpi=300, bbox_inches="tight")
plt.show()

## 6. Which features are redundant

Within Erdos-Renyi the expected clustering equals the edge probability, `C = p = k / (n - 1)`,
so density and clustering move together by construction. Watts-Strogatz separates them: `k` sets
the degree, `beta` sets the clustering. Any model containing both a density term and a clustering
term is therefore identified almost entirely off the WS realizations.

In [ ]:
FEATS = [
    "mean_deg",
    "std_deg",
    "deg_gini",
    "gcc",
    "avg_clustering",
    "mean_sp",
    "diameter",
    "modularity",
    "deg_assortativity",
    "centralization_deg",
    "centralization_btw",
    "algebraic_connectivity",
    "spectral_gap",
]

print("corr(mean_deg, gcc) within family\n")
for fam, sub in graphs.groupby("family"):
    print(
        f"  {fam:<16s} r = {sub['mean_deg'].corr(sub['gcc']):+.3f}   "
        f"gcc {sub['gcc'].min():.3f}-{sub['gcc'].max():.3f}   "
        f"deg {sub['mean_deg'].min():.1f}-{sub['mean_deg'].max():.1f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, fam in zip(axes, sorted(graphs["family"].unique())):
    Cm = graphs[graphs["family"] == fam][FEATS].corr()
    im = ax.imshow(Cm, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(FEATS)))
    ax.set_xticklabels(FEATS, rotation=90, fontsize=8)
    ax.set_yticks(range(len(FEATS)))
    ax.set_yticklabels(FEATS, fontsize=8)
    ax.set_title(f"{fam}  (n = {(graphs['family'] == fam).sum()} graphs)")
fig.colorbar(im, ax=axes, shrink=0.8, label="Pearson r")
fig.savefig(out_path / "feature_correlations.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Realizations that are structurally close but carry different family labels.
er = graphs[graphs["family"] == "erdos-renyi"]
ws = graphs[graphs["family"] == "watts-strogatz"]
pairs = [
    {
        "er": a["label"],
        "er_deg": round(a["mean_deg"], 1),
        "er_gcc": round(a["gcc"], 3),
        "ws": b["label"],
        "ws_deg": round(b["mean_deg"], 1),
        "ws_gcc": round(b["gcc"], 3),
        "d_deg": round(abs(a["mean_deg"] - b["mean_deg"]), 2),
        "d_gcc": round(abs(a["gcc"] - b["gcc"]), 3),
    }
    for _, a in er.iterrows()
    for _, b in ws.iterrows()
    if abs(a["mean_deg"] - b["mean_deg"]) < 1.0 and abs(a["gcc"] - b["gcc"]) < 0.05
]
print(f"ER/WS realizations within 1.0 mean degree and 0.05 clustering: {len(pairs)}")
pd.DataFrame(pairs).sort_values("d_gcc") if pairs else "none"

## 7. Degree distributions

One panel per realization. ER degrees are Poisson-shaped; WS degrees concentrate at `k` and
widen with `beta` (`watts-strogatz-06`, at `beta = 0.01`, is a near-unrewired ring lattice with
zero degree variance).

In [ ]:
nodes = (
    par.unique(subset=["graph_id", "agent_id"])
    .select(
        [
            "graph_id",
            "node_degree",
            "node_centrality_eigen",
            "node_centrality_between",
            "node_clustering",
        ]
    )
    .to_pandas()
    .merge(graphs[["graph_id", "short", "family", "mean_deg"]], on="graph_id")
)

order = graphs.sort_values(["family", "mean_deg"])["short"].tolist()


def small_multiples(column, xlabel, fname, bins=None):
    ncol = 8
    nrow = int(np.ceil(len(order) / ncol))
    fig, axes = plt.subplots(
        nrow, ncol, figsize=(1.9 * ncol, 1.5 * nrow), sharex=True, sharey=True
    )
    b = (
        bins
        if bins is not None
        else np.linspace(nodes[column].min(), nodes[column].max(), 18)
    )
    for ax, name in zip(axes.ravel(), order):
        sub = nodes[nodes["short"] == name]
        ax.hist(
            sub[column],
            bins=b,
            color=GRAPH_COLOR[sub["family"].iloc[0]],
            alpha=0.85,
            density=True,
        )
        ax.set_title(name, fontsize=7.5)
        ax.tick_params(labelsize=7)
    for ax in axes.ravel()[len(order) :]:
        ax.axis("off")
    fig.supxlabel(xlabel, fontsize=11)
    fig.supylabel("Density", fontsize=11)
    fig.tight_layout()
    fig.savefig(out_path / fname, dpi=300, bbox_inches="tight")
    plt.show()


small_multiples(
    "node_degree",
    "Node degree",
    "degree_distributions.pdf",
    bins=np.arange(0, nodes["node_degree"].max() + 2) - 0.5,
)

## 8. Centrality distributions

In [ ]:
small_multiples(
    "node_centrality_eigen", "Eigenvector centrality", "eigen_distributions.pdf"
)
small_multiples(
    "node_centrality_between", "Betweenness centrality", "betweenness_distributions.pdf"
)

## 9. Choosing a grouping variable

`graph_family` is a *generative* label, not a structural one. Section 6 found ten ER/WS pairs
that sit within 1.0 mean degree and 0.05 clustering of each other, so the label does not
partition the realizations cleanly. This section looks for a structural alternative.

A grouping variable is useful here if it satisfies two conditions at once:

1. **It carries structural variance** — it loads on the axes along which the 32 realizations
   differ.
2. **It cross-cuts the family label** — it encodes something ER/WS does not already say.
   Measured as `eta^2` of the feature on `family`: *low* is what we want. A feature with
   `eta^2 = 0.6` is mostly a restatement of the family name.

Features that score high on the first and low on the second are the candidates.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

FEATURES = [
    "mean_deg",
    "std_deg",
    "deg_gini",
    "gcc",
    "avg_clustering",
    "mean_sp",
    "diameter",
    "modularity",
    "deg_assortativity",
    "centralization_deg",
    "centralization_btw",
    "algebraic_connectivity",
    "spectral_gap",
]

X = graphs[FEATURES].astype(float)
Z = StandardScaler().fit_transform(X)

pca = PCA().fit(Z)
scores = pca.transform(Z)
components = pca.components_.copy()

# A principal component's sign is arbitrary. Orient PC1 so that it increases with
# density and PC2 so that it increases with clustering, otherwise the axis labels
# below describe the opposite of what the plot shows.
for j, anchor in [(0, "mean_deg"), (1, "gcc")]:
    if np.corrcoef(scores[:, j], X[anchor])[0, 1] < 0:
        scores[:, j] *= -1
        components[j] *= -1

graphs["PC1"], graphs["PC2"] = scores[:, 0], scores[:, 1]

evr = pca.explained_variance_ratio_
print(
    "explained variance:",
    np.round(evr[:4], 3),
    " cumulative:",
    np.round(np.cumsum(evr[:4]), 3),
)

loadings = (
    pd.DataFrame(components[:2].T, index=FEATURES, columns=["PC1", "PC2"])
    .assign(abs_PC1=lambda d: d["PC1"].abs())
    .sort_values("abs_PC1", ascending=False)
    .drop(columns="abs_PC1")
)
loadings.round(3)

**PC1 holds 62% of the structural variance and PC2 holds 20%.**

Both components are oriented below so that they increase with `mean_deg` and `gcc` respectively.

PC1 is a *density / integration* axis. Moving **right** (high PC1) means denser, shorter paths,
larger spectral gap and algebraic connectivity; moving **left** means sparser, longer paths and
higher modularity.

PC2 is a *cohesion* axis. Moving **up** means higher clustering and lower degree inequality,
which is what separates WS from ER at matched density.

The ER/WS label therefore tracks the **second** axis (20% of the variance), not the dominant one.

In [ ]:
def eta_squared(y: pd.Series, groups: pd.Series) -> float:
    """Share of a feature's variance explained by a categorical grouping."""
    grand = y.mean()
    ss_total = ((y - grand) ** 2).sum()
    ss_between = sum(len(v) * (v.mean() - grand) ** 2 for _, v in y.groupby(groups))
    return ss_between / ss_total if ss_total > 0 else np.nan


rank = pd.DataFrame(
    {
        "abs_r_PC1": [abs(np.corrcoef(X[f], graphs["PC1"])[0, 1]) for f in FEATURES],
        "eta2_family": [eta_squared(X[f], graphs["family"]) for f in FEATURES],
        "er_range": [
            f"{X[f][graphs['family'] == 'erdos-renyi'].min():.2f}-"
            f"{X[f][graphs['family'] == 'erdos-renyi'].max():.2f}"
            for f in FEATURES
        ],
        "ws_range": [
            f"{X[f][graphs['family'] == 'watts-strogatz'].min():.2f}-"
            f"{X[f][graphs['family'] == 'watts-strogatz'].max():.2f}"
            for f in FEATURES
        ],
    },
    index=FEATURES,
)

# High structural loading, low redundancy with the family label.
rank["score"] = rank["abs_r_PC1"] * (1 - rank["eta2_family"])
rank = rank.sort_values("score", ascending=False)
rank.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

ax = axes[0]
for fam, sub in graphs.groupby("family"):
    for is_origin, s in sub.groupby("is_origin"):
        ax.scatter(
            s["PC1"],
            s["PC2"],
            s=80 if is_origin else 60,
            marker="o" if is_origin else "^",
            facecolor=GRAPH_COLOR[fam] if is_origin else "none",
            edgecolor=GRAPH_COLOR[fam],
            linewidth=1.4,
            alpha=0.9,
            label=f"{fam} ({'original' if is_origin else 'extension'})",
        )
for _, r in graphs.iterrows():
    ax.annotate(
        r["short"],
        (r["PC1"], r["PC2"]),
        fontsize=6,
        xytext=(3, 3),
        textcoords="offset points",
        alpha=0.75,
    )
ax.set_xlabel(f"PC1 - density / integration ({evr[0]:.0%})")
ax.set_ylabel(f"PC2 - cohesion ({evr[1]:.0%})")
ax.legend(frameon=False, fontsize=8, loc="best")

ax = axes[1]
ax.scatter(rank["eta2_family"], rank["abs_r_PC1"], color=OKABE_ITO["blue"], s=45)
for f, r in rank.iterrows():
    ax.annotate(
        f,
        (r["eta2_family"], r["abs_r_PC1"]),
        fontsize=7,
        xytext=(4, -2),
        textcoords="offset points",
    )
ax.axvline(0.35, ls="--", lw=1, color="grey")
ax.set_xlabel(r"$\eta^2$ on family  (low = adds new information)")
ax.set_ylabel("|r| with PC1  (high = carries structural variance)")
ax.set_title("Candidate grouping variables", fontsize=11)

fig.tight_layout()
fig.savefig(out_path / "grouping_variable_selection.pdf", dpi=300, bbox_inches="tight")
plt.show()

### Does an unsupervised grouping find something new?

If clustering the realizations in structural space simply recovers ER/WS, there is no new
two-way split to be had and the answer has to be a different *kind* of grouping.

In [ ]:
print("silhouette by number of clusters (Ward linkage)\n")
labels = {}
for k in range(2, 7):
    labels[k] = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(Z)
    print(
        f"  k={k}:  silhouette={silhouette_score(Z, labels[k]):.3f}   "
        f"sizes={np.bincount(labels[k])}"
    )

graphs["cluster2"], graphs["cluster3"] = labels[2], labels[3]

print("\nk=2 against family:")
display(pd.crosstab(graphs["cluster2"], graphs["family"]))
agree = max(
    (graphs["cluster2"] == (graphs["family"] == "erdos-renyi")).mean(),
    (graphs["cluster2"] == (graphs["family"] == "watts-strogatz")).mean(),
)
print(f"k=2 reproduces the family label for {agree:.0%} of realizations\n")

print("k=3 against family:")
display(pd.crosstab(graphs["cluster3"], graphs["family"]))
display(
    graphs.groupby("cluster3")[["mean_deg", "gcc", "mean_sp", "modularity", "std_deg"]]
    .mean()
    .round(3)
)
for c, s in graphs.groupby("cluster3"):
    print(f"  cluster {c}: {sorted(s['short'].tolist())}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
dendrogram(
    linkage(Z, method="ward"),
    labels=graphs["short"].tolist(),
    leaf_font_size=7,
    color_threshold=0,
    above_threshold_color=OKABE_ITO["black"],
    ax=ax,
)
ax.set_ylabel("Ward distance")
plt.xticks(rotation=90)
fig.tight_layout()
fig.savefig(out_path / "structural_dendrogram.pdf", dpi=300, bbox_inches="tight")
plt.show()

### Candidate grouping: terciles of the leading cross-cutting features

For the top-ranked features, split the realizations into three bands and check that each band
draws from both families. A band drawn from a single family only restates the family label.

In [ ]:
def tercile_grouping(feature: str, n_bands: int = 3) -> pd.DataFrame:
    bands = pd.qcut(
        graphs[feature],
        n_bands,
        labels=[f"{feature[:12]}-{i + 1}" for i in range(n_bands)],
    )
    tab = pd.crosstab(bands, graphs["family"])
    tab["n"] = tab.sum(axis=1)
    tab["min_family_share"] = (
        tab[["erdos-renyi", "watts-strogatz"]].min(axis=1) / tab["n"]
    ).round(2)
    tab["range"] = [
        f"{g[feature].min():.2f}-{g[feature].max():.2f}"
        for _, g in graphs.groupby(bands, observed=True)
    ]
    return tab


for feature in rank.index[:4]:
    print(
        f"\n=== {feature}  (|r| PC1 = {rank.loc[feature, 'abs_r_PC1']:.2f}, "
        f"eta2 family = {rank.loc[feature, 'eta2_family']:.2f}) ==="
    )
    display(tercile_grouping(feature))

### Export

Keyed on `graph_id`, which is the join key in `agent_level_data.parquet`, so the downstream
analyses can merge these columns directly.

In [ ]:
grouping = graphs[
    [
        "graph_id",
        "label",
        "short",
        "family",
        "config",
        "master_seed",
        "PC1",
        "PC2",
        "cluster2",
        "cluster3",
        "mean_deg",
        "gcc",
        "mean_sp",
        "diameter",
        "modularity",
        "algebraic_connectivity",
        "n_runs",
    ]
].copy()

for feature in ["mean_deg", "mean_sp"]:
    grouping[f"{feature}_tercile"] = pd.qcut(
        graphs[feature], 3, labels=["low", "mid", "high"]
    )

grouping.to_csv(out_path / "graph_grouping.csv", index=False)
print(f"wrote {out_path / 'graph_grouping.csv'}  ({len(grouping)} realizations)")
grouping.head()

## 10. Grouping variables

**A two-way clustering does not improve on the family label.** Ward clustering at `k = 2`
reproduces the ER/WS split for 94% of realizations; the two realizations it moves are the
high-`beta` WS graphs that section 6 flagged as ER-like.

**Two measured axes describe the realizations better than one generative label.**

| axis | what it is | variable | `eta^2` on family |
|---|---|---|---|
| primary | density / integration (PC1, 62% of variance) | `mean_deg`, or `mean_sp` for a path-based reading | 0.16 / 0.23 |
| secondary | cohesion (PC2, 20% of variance) | `gcc` (or `avg_clustering`) | 0.60 |

- **`mean_deg` is the cleanest cross-cutting variable in the set.** It correlates 0.80 with PC1
  and has the lowest redundancy with family among the features that carry structural variance
  (`eta^2 = 0.16`); the two families span nearly the same range (ER 3.8-16.6, WS 4.0-16.0).
  Grouping on it produces bands drawn from both families.
- **`diameter`, `mean_deg` and `mean_sp` score almost identically** (0.676, 0.672, 0.660 at
  `n = 32`). `diameter` is integer-valued over 2-8 and is a single-path extremum, so one rewired
  edge can move it; `mean_deg` is continuous and stable, and enters the agent-level models as
  `node_degree_gm`.
- **`gcc` carries the cohesion axis.** Its `eta^2 = 0.60` means it largely *is* the ER/WS
  distinction, but as a measured quantity rather than a generative label, so the ten
  near-identical cross-family pairs land in the same group.
- **`std_deg`, `deg_gini`, `centralization_deg` and `modularity`** load heavily on PC1 but have
  `eta^2` of 0.56-0.67, so they mostly restate the family label.
- **`deg_assortativity` is uninformative here** (|r| with PC1 = 0.12, range -0.14 to 0.08). All
  32 realizations are effectively non-assortative.

Density enters the mixed models as a continuous term (`node_degree_gm`); the tercile tables
above are for plotting and for reporting group composition.

## 11. Summary

- **18 configs, 32 realizations**, all with run data (80 runs each: 4 scenarios x 20 statements).
- **The design is unbalanced**: `erdos-renyi` and `watts-strogatz` contribute eight realizations
  each, the 16 maximin-selected configs one apiece.
- **The eight realizations inside each base config are not interchangeable** (section 4).
- **Both families span the same density range**, so density is not confounded with the
  family label.
- **The family label tracks the second structural axis, not the first.** Density/integration is
  62% of the variance; cohesion, which ER/WS encodes, is 20%.
- **Density and clustering are collinear within ER** (`r = +0.98`) and independent within WS, so
  any model carrying both is identified off the WS realizations.